In [2]:
import itertools
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from sklearn.cluster import KMeans
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV
import elapid as ela
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

In [3]:
FEATURES = ["elevation", "slope","aspect","hcas"]

In [4]:
DATA_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()

In [5]:
SPECIES_CONFIGS=[
    {
        "species_name":"amytornis_purnelli",
        "training_csv":DATA_DIR/"amytornis_purnelli_training_matrix.csv",
        "raster_paths": {
            "elevation": DATA_DIR/"amytornis_purnelli_elevation.tif",
            "slope":DATA_DIR/"amytornis_purnelli_slope.tif",
            "aspect":DATA_DIR/"amytornis_purnelli_aspect.tif",
            "hcas":DATA_DIR/"amytornis_purnelli_hcas.tif",
        },
    },
    {
        "species_name":"atrichornis_rufescens",
        "training_csv":DATA_DIR/"atrichornis_rufescens_training_matrix.csv",
        "raster_paths": {
            "elevation": DATA_DIR/"atrichornis_rufescens_elevation.tif",
            "slope":DATA_DIR/"atrichornis_rufescens_slope.tif",
            "aspect":DATA_DIR/"atrichornis_rufescens_aspect.tif",
            "hcas":DATA_DIR/"atrichornis_rufescens_hcas.tif",
        },
    },
    {
        "species_name":"pedionomus_torquatus",
        "training_csv":DATA_DIR/"pedionomus_torquatus_training_matrix.csv",
        "raster_paths": {
            "elevation": DATA_DIR/"pedionomus_torquatus_elevation.tif",
            "slope":DATA_DIR/"pedionomus_torquatus_slope.tif",
            "aspect":DATA_DIR/"pedionomus_torquatus_aspect.tif",
            "hcas":DATA_DIR/"pedionomus_torquatus_hcas.tif",
        },
    },
    {
        "species_name":"pezoporus_occidentalis",
        "training_csv":DATA_DIR/"pezoporus_occidentalis_training_matrix.csv",
        "raster_paths": {
            "elevation": DATA_DIR/"pezoporus_occidentalis_elevation.tif",
            "slope":DATA_DIR/"pezoporus_occidentalis_slope.tif",
            "aspect":DATA_DIR/"pezoporus_occidentalis_aspect.tif",
            "hcas":DATA_DIR/"pezoporus_occidentalis_hcas.tif",
        },
    },
    {
        "species_name":"polytelis_alexandrae",
        "training_csv":DATA_DIR/"polytelis_alexandrae_training_matrix.csv",
        "raster_paths": {
            "elevation": DATA_DIR/"polytelis_alexandrae_elevation.tif",
            "slope":DATA_DIR/"polytelis_alexandrae_slope.tif",
            "aspect":DATA_DIR/"polytelis_alexandrae_aspect.tif",
            "hcas":DATA_DIR/"polytelis_alexandrae_hcas.tif",
        },
    },
    {
        "species_name":"leipoa_ocellata",
        "training_csv":DATA_DIR/"leipoa_ocellata_training_matrix.csv",
        "raster_paths": {
            "elevation": DATA_DIR/"leipoa_ocellata_elevation.tif",
            "slope":DATA_DIR/"leipoa_ocellata_slope.tif",
            "aspect":DATA_DIR/"leipoa_ocellata_aspect.tif",
            "hcas":DATA_DIR/"leipoa_ocellata_hcas.tif",
        },
    },
]     

In [6]:
N_SPATIAL_FOLDS = 5

In [7]:
RANDOM_STATE = 1234

In [8]:
OUTPUT_DIR = DATA_DIR/"outputs"

In [9]:
OUTPUT_DIR.mkdir(exist_ok=True)

# Step 1 - Loading the pre built training matrix

In [11]:
def sample_rasters_at_points(raster_paths: dict,coords: list[tuple[float,float]]) -> pd.DataFrame:
    out = {}
    for name,path in raster_paths.items():
        with rasterio.open(path) as src:
            out[name] = [val[0] for val in src.sample(coords)]
    return pd.DataFrame(out)

In [12]:
def load_training_matrix(csv_path: Path, features: list) -> pd.DataFrame: 
    df = pd.read_csv(csv_path)
    required = {"x_coord","y_coord","presence", *features}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{csv_path.name} is missing required columns: {missing}")
    df = df.dropna(subset = features)
    n_pos, n_neg = (df["presence"] == 1).sum(),(df["presence"] == 0).sum()
    print(f"Loaded {len(df)} rows ({n_pos} presence, {n_neg} background)")
    return df[["x_coord","y_coord","presence"] + features]

# Step 2 - Generate Background points from raster data

In [14]:
def generate_background_points(raster_paths:dict, n_points: int, random_state: int, block_rows: int = 512) -> pd.DataFrame:
    features = list(raster_paths.keys())
    rng = np.random.default_rng(random_state)
    ref_path = next(iter(raster_paths.values()))

    with rasterio.open(ref_path) as src:
        nodata = src.nodata
        height, width = src.height, src.width
        n_blocks = (height + block_rows - 1) // block_rows

        #Counting valid pixels per block
        block_counts = np.zeros(n_blocks, dtype = np.int64)
        for bi, row_start in enumerate(range(0, height, block_rows)):
            rows = min(block_rows, height - row_start)
            window = Window(0,row_star, width, rows)
            band = src.read(1, window = window)
            valid_mask = np.isinfinite(band) if nodata is None else (band != nodata) & np.isfinite(band)
            block_counts[bi] = valid_mask.sum()
        total_valid = block_counts.sum()
        if total_valid ==0:
            raise ValueError(f"No Valid (non-nodata) pixels found in {ref_path}")
            if total_valid ==0:
                raise ValueError(f"No valid (non-nodata) pixels found in {ref-path}")
            if total_valid < n_points:
                print (f"Warning: only {total_valid} valid pixels available, requested {n_points}")
                n_points = int(total_valid)

            raw_alloc = block_counts/total_valid * n_points
            alloc = np.floor(raw_alloc).astype(np.int64)
            shortfall = n_points - alloc.sum()
            if shortfall > 0:
                remainders = raw_alloc - alloc
                top_up = np.argsort(-remainders)[:shortfall]
                alloc[top_up] += 1
            alloc = np.minimum(alloc, block_counts)

            # Vectorising sample within each block
            all_rows, all_cols = [], []
            for bi, row_start in enumerate(range(0,height, block_rows)):
                if alloc[bi]==0:
                    continue
                rows = min(block_rows, height - row_start)
                window = Window(0,row_start, width, rows)
                band = src.read(1, window=window)
                valid_mask = np.isfinite(band) if nodata is None else (band!=nodata) & np.isfinite(band)
                local_rows,local_cols = np.where(valid_mask)
                
                chosen=rng.choice(len(local_rows),size=int(alloc[bi],replace =False))
                all_rows.append(local_rows[chosen] +row_start)
                all_cols.append(local_cols[chosen])
                
            chosen_rows = np.concatenate(all_rows)
            chosen_cols = np.concatenate(all_cols)
            xs,ys = rasterio.transform.xy(src.transform,chosen_rows,chosen_cols)
        coords = list(zip(xs,ys))
        bg = sample_rasters_at_points(raster_paths,coords)
        bg["x_coord"] = xs
        bg["y_coord"] = ys
        bg["presence"] = 0
        bg=bg.dropna(subset=features)
        return bg[["x_coord","y_coord","presence"] + features]

# Step 3 - Spatial CV Folds

In [16]:
def assign_spatial_folds(df: pd.DataFrame, n_folds: int, random_state: int) -> pd.DataFrame:
    presence_sites = df[df["presence"] == 1].drop_duplicates(subset=["x_coord","y_coord"])
    background_sites = df[df["presence"] == 0].drop_duplicates(subset = ["x_coord","y_coord"])
    
    kmeans = KMeans(n_clusters = n_folds, n_init = 50, random_state = random_state)
    presence_sites = presence_sites.copy()
    presence_sites["fold"] = kmeans.fit_predict(presence_sites[["x_coord","y_coord"]])
    background_sites = background_sites.copy()
    background_sites["fold"] = kmeans.predict(background_sites[["x_coord","y_coord"]])
    
    unique_sites = pd.concat([presence_sites, background_sites],ignore_index = True)
    print(pd.crosstab(unique_sites["fold"],unique_sites["presence"]).rename(columns={0:"Background",1: "Presence"}))
    
    df=pd.merge(df,unique_sites[["x_coord","y_coord","fold"]],on=["x_coord","y_coord"],how="left")
    return df

In [17]:
def spatial_cv_splits(df:pd.DataFrame,n_folds:int):
    idx=df.index.to_numpy()
    for test_fold in range(n_folds):
        test_mask = (df["fold"]==test_fold).to_numpy()
        yield idx[~test_mask],idx[test_mask]

# Step 4 - Hyperparameter tuning

In [19]:
def tune_lightgbm(X,y,cv_splits):
    grid = {
        "n_estimators":[50,150],
        "max_depth":[2,3,-1],
        "learning_rate":[0.01,0.1],
        "num_leaves":[7,31],
        "min_child_samples":[5,10],
    }
    search = GridSearchCV(
    lgb.LGBMClassifier(random_state = RANDOM_STATE, verbosity =-1),
    grid, cv = cv_splits, scoring = "roc_auc", n_jobs =-1,
    )
    search.fit(X,y)
    return search.best_params_, search.best_score_

In [20]:
def tune_xgboost(X, y, cv_splits):
    grid = {"n_estimators": [50,150],
            "max_depth": [1,2,3],
            "learning_rate": [0.01, 0.1],
            "subsample": [0.8, 1.0],
            "colsample_bytree":[0.8,1.0],
            "reg_lambda": [1,10],
           }
    search = GridSearchCV(
    xgb.XGBClassifier(random_state = RANDOM_STATE, eval_metric = "logloss"), 
    grid, cv=cv_splits, scoring = "roc_auc", n_jobs = -1,
    )
    search.fit(X,y)
    return search.best_params_, search.best_score_

In [21]:
def tune_catboost(X, y, cv_splits):
    grid = {
        "iterations": [50,150],
        "depth": [2,3,4],
        "learning_rate": [0.01,0.1],
        "l2_leaf_reg": [1,10],
    }
    search = GridSearchCV(
        CatBoostClassifier(random_state=RANDOM_STATE, verbose=0),
        grid, cv=cv_splits, scoring = "roc_auc", n_jobs =-1,
    )
    search.fit(X,y)
    return search.best_params_, search.best_score_

In [22]:
def tune_maxent(X, y, cv_splits):
    grid = {
        "feature_types": [
            ["linear", "hinge"], ["hinge"],
            ["linear", "quadratic"],
        ],
        "beta_multiplier": [1.0, 4.0, 8.0],
        "class_weights": [100, "balanced"],
    }
    keys = list(grid.keys())
    best_score, best_params = -np.inf, None
    n_fold_errors=0
    for combo in itertools.product(*grid.values()):
        params = dict(zip(keys, combo))
        fold_aucs = []
        for train_idx, test_idx in cv_splits:
            X_train, y_train = X.loc[train_idx], y.loc[train_idx]
            X_test, y_test = X.loc[test_idx], y.loc[test_idx]
            if y_test.nunique() <2:
                continue
            try:
                model =ela.MaxentModel(transform = "cloglog", random_state = RANDOM_STATE, **params)
                model.fit(X_train, y_train)
                preds = np.asarray(model.predict(X_test)).ravel()
                if not np.all(np.isfinite(preds)):
                    n_fold_errors +=1
                    continue
                fold_aucs.append(roc_auc_score(y_test, preds))
            except Exception:
                n_fold_errors +=1
                continue
        if fold_aucs:
            mean_auc = np.mean(fold_aucs)
            if mean_auc > best_score:
                best_score, best_params = mean_auc, params
    if n_fold_errors > 0:
        print(f" (MaxEnt: skipped {n_fold_errors} fold/param fits due to degenerate folds --"
              f"likely a fold with very few presence points)")
    if best_params is None:
        raise ValueError(
            "MaxEnt could not be fit on ANY fold/paramter combination for this species --"
            "every fold was too degenerate (e.g. too few presence points per fold). Consider"
            "using fewer spatial folds (reduce N_SPATIAL_FOLDS) for this species."
        )
            
    return best_params, best_score

# Step 5 - Ensemble Model

In [24]:
def get_per_fold_predictions(model_fns: dict, X, y, cv_splits):
    fold_results = []
    n_skipped_folds = 0
    for train_idx, test_idx in cv_splits:
        X_train, y_train = X.loc[train_idx], y.loc[train_idx]
        X_test, y_test = X.loc[test_idx], y.loc[test_idx]
        if y_test.nunique() < 2:
            continue
        preds = {}
        fold_ok = True
        for name, fn in model_fns.items():
            try:
                m=fn()
                m.fit(X_train, y_train)
                if name == "maxent":
                    p = np.asarray(m.predict(X_test)).ravel()
                else:
                    p = m.predict_proba(X_test)[:, 1]
                if not np.all(np.isfinite(p)):
                    fold_ok = False
                    break
                preds[name] = p
            except Exception:
                fold_ok = False
                break
        if fold_ok:
            fold_results.append((y_test.values, preds))
        else:
            n_skipped_folds +=1
    if n_skipped_folds > 0:
        print(f" (Ensemble: skipped {n_skipped_folds} degenerate fold(s) entirely --"
              f"a model could not produce valid predictions on them)")
    if not fold_results:
        raise ValueError(
            "No fold produced valid predictions from every model -- cannot build an "
            "ensemble for this species. Consider reducing N_SPATIAL_FOLDS."
        )
    return fold_results

In [25]:
def evaluate_blend(fold_results, weights: dict):
    total = sum(weights.values())
    weights = {k: v/total for k,v in weights.items()}
    aucs=[]
    for y_test, preds in fold_results:
        blended = sum(weights[name] * preds[name] for name in weights)
        aucs.append(roc_auc_score(y_test, blended))
    return np.mean(aucs), np.std(aucs)

In [26]:
def search_ensemble_weights(fold_results, model_names, n_trials=400, random_state = RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    best_score, best_weights = -np.inf, None
    for _ in range(n_trials):
        raw = rng.dirichlet(np.ones(len(model_names)))
        weights = dict(zip(model_names, raw))
        mean_auc, std_auc = evaluate_blend(fold_results, weights)
        if mean_auc > best_score:
            best_score, best_weights = mean_auc, weights
    return best_weights, best_score

# Fitting the Ensemble Model and GeoTIFF output

In [28]:
def predict_suitability_raster(models: dict, weights: dict, raster_paths: dict, features: list, out_path: Path, block_rows: int = 256):
    ref_path = next(iter(raster_paths.values()))
    with rasterio.open(ref_path) as ref:
        profile = ref.profile.copy()
        height, width = ref.height, ref.width
    profile.update(dtype = "float32", count = 1, nodata=-9999.0)

    srcs = {name: rasterio.open(path) for name, path in raster_paths.items()}
    try:
        with rasterio.open(out_path, "w", **profile) as dst:
            for row_start in range(0, height, block_rows):
                rows = min(block_rows, height - row_start)
                window = Window(0, row_start, width, rows)

                block_data ={}
                nodata_mask = np.zeros((rows,width), dtype = bool)
                for name in raster_paths:
                    arr = srcs[name].read(1,window = window)
                    nd = srcs[name].nodata
                    if nd is not None:
                        nodata_mask |= (arr ==nd)
                    nodata_mask |= ~np.isfinite(arr)
                    block_data[name] = arr
                flat = {name: block_data[name].ravel() for name in raster_paths}

                X_block = pd.DataFrame(flat)[features]

                total_w = sum(weights.values())
                blended = np.zeros(len(X_block), dtype="float64")
                for name, w in weights.items():
                    if name == "maxent":
                        preds = np.asarray(models[name].predict(X_block)).ravel()
                    else:
                        preds = models[name].predict_proba(X_block)[:, 1]
                    blended += (w/total_w) * preds
                out_block = blended.reshape(rows,width).astype("float32")
                out_block[nodata_mask.reshape(rows,width)] = -9999.0
                dst.write(out_block, 1, window=window)
    finally:
        for s in srcs.values():
            s.close()
    print(f"Saved suitability raster: {out_path}")

# Per Species Pipeline

In [30]:
def run_pipeline_for_species(config: dict) -> dict:
    species_name = config["species_name"]
    training_csv = config["training_csv"]
    raster_paths = config["raster_paths"]
    features = FEATURES

    print(f"\n{'=' * 70}\n=== Species Distribution Model: {species_name} ===\n{'=' * 70}\n")

    #Step 1: Load the pre-built training matrix
    df = load_training_matrix(training_csv, features)

    #Step 2: Spatial folds
    print("\nAssigning spatial folds...")
    df=assign_spatial_folds(df, N_SPATIAL_FOLDS, RANDOM_STATE)
    cv_splits = list(spatial_cv_splits(df, N_SPATIAL_FOLDS))

    X,y = df[features], df["presence"]

    #Step 3: Tune each model
    print("\n Tuning LightGBM")
    lgbm_params, lgbm_auc = tune_lightgbm(X,y,cv_splits)
    print(f"Best: {lgbm_params} AUC={lgbm_auc:.3f}")

    print("\n Tuning XGBoost")
    xgb_params, xgb_auc = tune_xgboost(X,y,cv_splits)
    print(f"Best: {xgb_params} AUC={xgb_auc:.3f}")

    print("\n Tuning CatBoost")
    cb_params, cb_auc = tune_catboost(X,y,cv_splits)
    print(f"Best: {cb_params} AUC={cb_auc:.3f}")

    print("\n Tuning MaxEnt")
    maxent_params, maxent_auc = tune_maxent(X,y,cv_splits)
    print(f"Best: {maxent_params} AUC={maxent_auc:.3f}")

    individual_results = pd.DataFrame([
        {"model": "MaxEnt", "cv_auc": maxent_auc},
        {"model": "LightGBM", "cv_auc": lgbm_auc},
        {"model": "XGBoost", "cv_auc": xgb_auc},
        {"model": "CatBoost", "cv_auc": cb_auc},
    ]).sort_values("cv_auc",ascending=False)
    print(f"\n=== [{species_name}] Individual model comparison ===")
    print(individual_results.to_string(index=False))

    #Step 4: Ensemble combinations
    model_fns = {
        "maxent": lambda:ela.MaxentModel(transform="cloglog", random_state=RANDOM_STATE, **maxent_params),
        "lgbm":lambda:lgb.LGBMClassifier(random_state=RANDOM_STATE,verbosity=-1, **lgbm_params),
        "xgb":lambda: xgb.XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss", **xgb_params),
        "catboost":lambda: CatBoostClassifier(random_state=RANDOM_STATE, verbose=0, **cb_params),
    }

    print("\nComputing per-fold predictions for all 4 models...")
    fold_results = get_per_fold_predictions(model_fns, X, y, cv_splits)

    print(f"\n=== [{species_name}] Trying model combinations ===")
    combo_results =[]
    all_names = list(model_fns.keys())
    for r in range(2, len(all_names) +1):
        for combo in itertools.combinations(all_names, r):
            weights, score = search_ensemble_weights(fold_results, combo, n_trials=100)
            combo_results.append({
                "combination": "+".join(combo),
                "n_models":r,
                "cv_auc": score,
                "weights": {k:round(v,3) for k,v in weights.items()},
            })
            print(f" {' + '.join(combo):40s} AUC={score:.3f} weights={weights}")
    
    combo_df = pd.DataFrame(combo_results).sort_values("cv_auc",ascending=False)
    print(f"\n=== [{species_name}] Top 5 combinations ===")
    print(combo_df.head(5).to_string(index=False))

    best_combo = combo_df.iloc[0]
    print(f"\nBest Combination for {species_name}: {best_combo['combination']} (AUC={best_combo['cv_auc']:.3f})")

    #Step 5: Refit best combination on full data
    best_weights = best_combo["weights"]
    print(f"\nRefitting {list(best_weights.keys())} on full dataset...")
    final_models ={}
    for name in best_weights:
        m=model_fns[name]()
        m.fit(X,y)
        final_models[name]=m

    import joblib
    model_path=OUTPUT_DIR/f"{species_name}_ensemble.joblib"
    joblib.dump(
        {"models": final_models, "weights": best_weights, "features": features}, model_path,)
    print(f"Saved model bundle: {model_path}")

    #Step 6: Raster export
    missing_rasters = [name for name, path in raster_paths.items() if not path.exists()]
    raster_out_path = None
    if missing_rasters:
        print(
            f"\nSkipping raster export for {species_name}: missing raster file(s)"
            f"{missing_rasters}. Model is tuned and saved -- rerun once those TIFs"
            f"are in this folder to generate the suitability map."
        )
    else:
        print(f"\nGenerating suitability raster for {species_name}...")
        raster_out_path = OUTPUT_DIR/f"{species_name}_suitability.tif"
        predict_suitability_raster(final_models, best_weights, raster_paths, features, raster_out_path)

    return {
        "species": species_name,
        "n_rows": len(df),
        "best_combination": best_combo["combination"],
        "best_cv_auc": best_combo["cv_auc"],
        "best_weights": best_weights,
        "individual_results": individual_results,
        "model_path": model_path,
        "raster_path":raster_out_path,
    }

# BATCH RUNNER

In [32]:
def main():
    all_results = []
    for config in SPECIES_CONFIGS:
        try:
            result = run_pipeline_for_species(config)
            all_results.append(result)
        except FileNotFoundError as e:
            print(f"\n!!! Skipping {config['species_name']}: missing file - {e}\n")
        except Exception as e:
            print(f"\n!!! {config['species_name']} failed: {e}\n")
    print(f"\n{'=' * 70}\n===FINAL SUMMARY - ALL SPECIES ===\n{'=' * 70}")
    summary_df = pd.DataFrame([
        {
            "species": r["species"],
            "n_rows": r["n_rows"],
            "best_combination": r["best_combination"],
            "best_cv_auc": r["best_cv_auc"],
        }
        for r in all_results
    ]).sort_values("best_cv_auc",ascending = False)
    print(summary_df.to_string(index=False))

    summary_df.to_csv(OUTPUT_DIR / "all_species_summary.csv", index=False)
    print(f"\n Saved summary: {OUTPUT_DIR / 'all_species_summary.csv'}")

In [33]:
if __name__ == "__main__":
    main()


=== Species Distribution Model: amytornis_purnelli ===

Loaded 6426 rows (3213 presence, 3213 background)

Assigning spatial folds...
presence  Background  Presence
fold                          
0                436       772
1                649       288
2                387        54
3               1341       172
4                400        18

 Tuning LightGBM
Best: {'learning_rate': 0.1, 'max_depth': 3, 'min_child_samples': 5, 'n_estimators': 150, 'num_leaves': 7} AUC=0.944

 Tuning XGBoost
Best: {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 150, 'reg_lambda': 1, 'subsample': 1.0} AUC=0.944

 Tuning CatBoost
Best: {'depth': 3, 'iterations': 150, 'l2_leaf_reg': 10, 'learning_rate': 0.1} AUC=0.951

 Tuning MaxEnt
Best: {'feature_types': ['linear', 'quadratic'], 'beta_multiplier': 1.0, 'class_weights': 'balanced'} AUC=0.946

=== [amytornis_purnelli] Individual model comparison ===
   model   cv_auc
CatBoost 0.950862
  MaxEnt 0.945987
 XGBoost 0.94

/Users/missbhat/Monash/IE/Onboarding/TA27_PathEase/.conda/lib/python3.12/site-packages/elapid/models.py:812: RuntimeWarning: divide by zero encountered in log
  return -np.sum(scaled * np.log(scaled))
/Users/missbhat/Monash/IE/Onboarding/TA27_PathEase/.conda/lib/python3.12/site-packages/elapid/models.py:812: RuntimeWarning: invalid value encountered in multiply
  return -np.sum(scaled * np.log(scaled))


 (MaxEnt: skipped 1 fold/param fits due to degenerate folds --likely a fold with very few presence points)
Best: {'feature_types': ['linear', 'quadratic'], 'beta_multiplier': 1.0, 'class_weights': 100} AUC=0.735

=== [pedionomus_torquatus] Individual model comparison ===
   model   cv_auc
 XGBoost 0.760296
CatBoost 0.758569
LightGBM 0.756861
  MaxEnt 0.735481

Computing per-fold predictions for all 4 models...


/Users/missbhat/Monash/IE/Onboarding/TA27_PathEase/.conda/lib/python3.12/site-packages/elapid/models.py:812: RuntimeWarning: divide by zero encountered in log
  return -np.sum(scaled * np.log(scaled))
/Users/missbhat/Monash/IE/Onboarding/TA27_PathEase/.conda/lib/python3.12/site-packages/elapid/models.py:812: RuntimeWarning: invalid value encountered in multiply
  return -np.sum(scaled * np.log(scaled))


 (Ensemble: skipped 1 degenerate fold(s) entirely --a model could not produce valid predictions on them)

=== [pedionomus_torquatus] Trying model combinations ===
 maxent + lgbm                            AUC=0.776 weights={'maxent': np.float64(0.17937226129758188), 'lgbm': np.float64(0.8206277387024181)}
 maxent + xgb                             AUC=0.777 weights={'maxent': np.float64(0.1727334527560751), 'xgb': np.float64(0.827266547243925)}
 maxent + catboost                        AUC=0.782 weights={'maxent': np.float64(0.27278820002469145), 'catboost': np.float64(0.7272117999753085)}
 lgbm + xgb                               AUC=0.776 weights={'lgbm': np.float64(0.4322876089019152), 'xgb': np.float64(0.5677123910980848)}
 lgbm + catboost                          AUC=0.781 weights={'lgbm': np.float64(0.0015002631889651048), 'catboost': np.float64(0.9984997368110349)}
 xgb + catboost                           AUC=0.781 weights={'xgb': np.float64(0.0015002631889651048), 'catboost': n